## Workshop Lab 5b: Capstone Challenge - New Mexico Wildlife Proximity Analysis
**The mission:** Put your new coding knowledge to the test! For this notebook, you will:
- Fetch iNaturalist observations on **Mule Deer** in New Mexico
- Clean and export that data as a csv
- Convert the CSV into a point feature class
- Use Arcade expressions to visualize deer based on proximity to Black Bear sightings.

**How to approach this lab:**
You don't need to write all code from scratch! Professional developers rarely write code from memory. Instead they find working examples, copy them, and adapt the variables.
- **Reuse existing code:** Look back at **Notebook 3**, **Notebook 4**, **Notebook 5**, and your `.py` script files.
- **Follow the references:** Each step below tells you exactly which notebook cell or script file to open and copy from.
- **Adapt the parameters:** Update species names, place IDs, and file paths to match New Mexico Mule Deer.

*If you get stuck, check `05b_Solutions.ipynb` in the `02_notebooks` folder.*

### Step 1: Collect and Clean the Data
Query the iNaturalist API for **Mule Deer** in **New Mexico**, clean the observations, and save them as a CSV.

**Where to get your code from:**
Open Notebook 5 (05_SharingAndAPIs.ipynb), look at the black bear API fetch. Copy the requests.get() structure, the parsing loop, and the DataFrame creation. Stop before the arcpy conversion part.
<details>
<summary><b> **What to adapt/check:** </b></summary>

- **`taxon_name`**: Change `"Ursus americanus"` to `"Odocoileus hemionus"` (Mule Deer).
- **`place_id`**: Use `9` (New Mexico).
- **`per_page`**: Set to `200`.
- **Output File Path**: Save your cleaned CSV to `"../04_data/clean/mule_deer_newmexico.csv"`.
</details>

In [ ]:
# Complete the code below:

import requests
import pandas as pd
import os

# Define API endpoint and request parameters
url = "https://api.inaturalist.org/v1/observations"
params = {
    "taxon_name": ____,      # the scientific name for Mule Deer
    "place_id": ____,        # check the place_id the black bear fetch used in Notebook 5
    "per_page": ____,        # how many observations to ask for
    "verifiable": "true"
}

# Make the API request and parse the JSON response
response = requests.get(url, params=____)
data = response.json()
observations = data.get(____, [])    # which key holds the list of observations?

# Pull out the pieces we want from each observation
records = []
for obs in ____:
    geo = obs.get("geojson")
    if geo and "coordinates" in geo:
        lon, lat = geo["coordinates"]    # coordinates arrive as [longitude, latitude]
        records.append({
            "id": obs.get("id"),
            "observed_on": obs.get("observed_on"),
            "scientific_name": obs.get("species_guess"),
            "common_name": obs.get("common_name"),
            "latitude": ____,
            "longitude": ____
        })

df = pd.DataFrame(records)
df_clean = df.dropna(subset=[____, ____]).copy()   # drop rows with no coordinates

os.makedirs("../04_data/clean", exist_ok=True)
output_csv = ____    # where should the cleaned CSV go? (see the instructions above)
df_clean.to_csv(output_csv, index=False)
print(f"Success! Saved {len(df_clean)} records to {output_csv}")

### Step 2: Convert CSV into a Feature Class using ArcPy

Push your exported Mule Deer CSV into your `IntroToPythonAndArcade.gdb` as a Point Feature Class using `arcpy.management.XYTableToPoint`.

**Where to copy your code from:**
1. Open Notebook 3 (`03_data_wrangling.ipynb`), look at the top of the Cleaning Script chunk. Look at how input_path and output_path are defined as direct string variables (instead of using tool parameters like GetParameterAsText). Copy this.
2. Open your standalone Python script **`05_scripts/XYtoPoint.py`** (first used in **Notebook 3**, under the *Second Script Tool: XYtoPoint.py* section).
3. Copy the `arcpy.management.XYTableToPoint()` function and spatial reference setup.

<details>
<summary><b> **What to adapt/check:** </b></summary>

- **Input CSV (`in_table`)**: Point to `"../04_data/clean/mule_deer_newmexico.csv"`.
- **Output Feature Class (`out_feature_class`)**: Point directly to your project GDB: `"IntroToPythonAndArcade.gdb/Mule_Deer_Points"`.
- **Script Parameters vs. Direct Variables**: The script tool used `arcpy.GetParameterAsText()`. For this notebook cell, replace those with explicit string variables for `input_csv`, `gdb_path`, and `output_fc`.
</details>

In [ ]:
# Complete the code below:

import arcpy
import os

# Build the full path to your cleaned CSV from Step 1 (same pattern as gdb_path below)
input_csv = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "____", "____"))
gdb_path = os.path.abspath(os.path.join(os.getcwd(), "..", "03_pro_project", "IntroToPythonAndArcade.gdb"))
output_fc = os.path.join(gdb_path, "____")     # a name for the new feature class

# Coordinate field names and spatial reference (WGS 1984 / EPSG 4326)
x_field = "____"      # which column is x: longitude or latitude?
y_field = "____"
spatial_ref = arcpy.SpatialReference(____)

# If the feature class already exists, delete it so this cell can be rerun
if arcpy.Exists(____):
    arcpy.management.Delete(output_fc)

# Create the feature class
arcpy.management.XYTableToPoint(
    in_table=____,
    out_feature_class=____,
    x_field=x_field,
    y_field=y_field,
    coordinate_system=____
)

print(f"Successfully created point feature class: {output_fc}")

### Step 3: Symbology & Arcade Proximity in ArcGIS Pro

Switch over to **ArcGIS Pro**, add `Mule_Deer_Points` and `smblack_bears_cleaned` to your map, and apply Arcade expressions.

**Where to copy your expressions from:**
1. Open **Notebook 4 (`04_Arcade.ipynb`)**.
2. **Distance / Size Scaling:** Look at the chunk where we Vary Symbology by Attribute / Age of Observation.
3. **Popup Spatial Intersection:** Look at `Roadrunners500m`, which uses `FeatureSetByName()`, `BufferGeodetic()`, and `Intersects()`.

**What to adapt in ArcGIS Pro:**
- **Layer Context**: In the popup expression, update `FeatureSetByName($map, "all_roadrunners_cleaned_XYtoPoint")` to target your bear layer: `"smblack_bears_cleaned"`.
- **Buffer Distance**: Adjust the `BufferGeodetic` distance (e.g., 1000 meters) to evaluate deer proximity to known bear sightings.